# Ph.4 — адаптация на MegaVul

Измеряет, что даёт дообучение на целевом домене и **сколько из этого приходится
на LoRA**. Прежний прогон Ph.4 менял две вещи разом и весь прирост приписал
адаптерам, которые в том режиме не обучались вовсе.

Два плеча отличаются ровно одним — двигаются адаптеры или нет:

| плечо | флаги | обучается |
|---|---|---|
| **B0** | `--freeze_encoder --freeze_lora` | GNN + головы, ~1.07M |
| **B1** | `--freeze_encoder` | то же + LoRA, ~1.36M |

**Вклад LoRA = B1 − B0.**

Вариант «одно плечо без `--use_lora`» не годится: слим-чекпоинт несёт тензоры
`lora_A`/`lora_B`, и модель без LoRA его не примет — архитектура в обоих плечах
обязана быть одной.

## Что нужно на Drive заранее

```
MyDrive/gnn-regvd/
├── megavul_simple.json                      <- 1.2 ГБ, из архива MegaVul
└── lora_contextual/checkpoint-best-acc/
    ├── model.bin                            <- стартовые веса (Devign)
    └── model_config.json                    <- обязателен: из него берётся архитектура
```


## 1. Окружение

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Нет GPU: Runtime -> Change runtime type -> T4 GPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/gnn-regvd'
SRC   = f'{DRIVE}/megavul_simple.json'
INIT  = f'{DRIVE}/lora_contextual/checkpoint-best-acc/model.bin'

import os
for path in (SRC, INIT, INIT.replace('model.bin', 'model_config.json')):
    print(('OK   ' if os.path.exists(path) else 'НЕТ  ') + path)

## 2. Код

In [ ]:
%cd /content
BRANCH = "fix/lora-gradient-flow-and-module0"
!git clone -b {BRANCH} https://github.com/deccersw/GNN-ReGVD-Plus.git 2>/dev/null || (cd GNN-ReGVD-Plus && git pull)
%cd /content/GNN-ReGVD-Plus
!git log --oneline -1
# Ph.4 требует правок, которых нет в main: --init_from, --freeze_lora,
# --freeze_encoder, --pos_weight, eval_pr_auc, dataset/megavul.py.
!git log --oneline -1 -- dataset/megavul.py || echo "ВЕТКА СТАРАЯ: dataset/megavul.py отсутствует"
!pip install -q "transformers>=4.30" faiss-cpu

## 3. Данные

Разбор идёт с локального диска, а не с примонтированного Drive: чтение 1.2 ГБ
через FUSE медленнее в разы. Готовые сплиты копируются обратно на Drive, чтобы
следующие пять прогонов начинались сразу с них — пересборка занимает ~10 минут
и на каждый прогон её повторять незачем.

In [ ]:
import os, shutil, time

DATA  = '/content/data/megavul'
CACHE = f'{DRIVE}/megavul_splits'
NEEDED = ['megavul_train_sub.jsonl', 'megavul_valid_15k.jsonl', 'megavul_test.jsonl']
os.makedirs(DATA, exist_ok=True)

if all(os.path.exists(f'{CACHE}/{n}') for n in NEEDED):
    print("Готовые сплиты найдены на Drive, пересборка не нужна")
    for n in NEEDED:
        shutil.copy(f'{CACHE}/{n}', f'{DATA}/{n}')
else:
    t = time.time()
    if not os.path.exists('/content/megavul_simple.json'):
        print("копирую исходник с Drive на локальный диск...")
        shutil.copy(SRC, '/content/megavul_simple.json')
        print(f"  {time.time()-t:.0f}s")

    steps = [
        f'python dataset/megavul.py extract --src /content/megavul_simple.json --out-dir {DATA}',
        f'python dataset/megavul.py split --out-dir {DATA}',
        f'python dataset/megavul.py subsample --split train --neg-ratio 4 --out-dir {DATA} --out {DATA}/megavul_train_sub.jsonl',
        f'python dataset/megavul.py subsample --split valid --size 15000 --out-dir {DATA} --out {DATA}/megavul_valid_15k.jsonl',
    ]
    for step in steps:
        print('\n$', step)
        !{step}

    os.makedirs(CACHE, exist_ok=True)
    for n in NEEDED:
        shutil.copy(f'{DATA}/{n}', f'{CACHE}/{n}')
    print(f"готово за {time.time()-t:.0f}s, сплиты сохранены в {CACHE}")

### Проверка воспроизводимости

Разбор и разбиение детерминированы (сид 42). Если числа ниже не совпадут с
ожидаемыми — данные другие, и сравнивать результаты с локальными измерениями
нельзя.

In [ ]:
import json, collections

EXPECTED = {
    'megavul_train_sub.jsonl': (58395, 11679),
    'megavul_valid_15k.jsonl': (15000,   748),
    'megavul_test.jsonl':      (49707,  2446),
}
ok = True
for name, (n_exp, p_exp) in EXPECTED.items():
    rows = [json.loads(l) for l in open(f'{DATA}/{name}')]
    n, p = len(rows), sum(r['target'] for r in rows)
    good = (n, p) == (n_exp, p_exp)
    ok &= good
    print(f"{'OK ' if good else 'РАЗОШЛОСЬ'} {name:26s} {n:6d} функций, "
          f"{p:5d} уязвимых ({p/n:.2%})   ожидалось {n_exp}/{p_exp}")
assert ok, "Выборка не совпала с эталоном — дальше идти нельзя"

# Утечки между train и test быть не должно: разбиение идёт по CVE.
cves = {s: {json.loads(l)['cve'] for l in open(f'{DATA}/megavul_{s}.jsonl')}
        for s in ('train', 'test')}
print("\nобщих CVE между train и test:", len(cves['train'] & cves['test']))

## 4. Проверка перед долгим запуском

Двадцать секунд, которые экономят часы: убеждаемся, что стартовый чекпоинт
читается, архитектура берётся из его сидкара, а каждый обучаемый тензор
действительно в графе автограда.

In [ ]:
%cd /content/GNN-ReGVD-Plus/code
import torch, logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
from transformers import RobertaConfig, RobertaForSequenceClassification, RobertaTokenizer
from model import GNNReGVD, check_gradient_flow, load_checkpoint_weights, load_model_config
from losses import SupervisedContrastiveLoss

sidecar = load_model_config(INIT)
print("архитектура из чекпоинта:", sidecar)
assert sidecar and sidecar['use_lora'] and sidecar['encoder_mode'] == 'contextual', \
    "стартовый чекпоинт не тот: ожидается contextual + LoRA"

class A: pass
args = A()
for k, v in sidecar.items(): setattr(args, k, v)
args.freeze_encoder, args.freeze_lora, args.pos_weight = True, False, 4.0

tok = RobertaTokenizer.from_pretrained('microsoft/graphcodebert-base')
cfg = RobertaConfig.from_pretrained('microsoft/graphcodebert-base'); cfg.num_labels = 1
enc = RobertaForSequenceClassification.from_pretrained('microsoft/graphcodebert-base', config=cfg)
model = GNNReGVD(enc, cfg, tok, args).cuda()
load_checkpoint_weights(INIT, model, torch.device('cuda'), args)

# Настоящая цель обучения, а не только BCE: эмбеддинг-голова обучается
# контрастивным членом, и по одному классификационному лоссу она законно
# выглядит отсоединённой.
contrastive = SupervisedContrastiveLoss(temperature=0.07)
ids = torch.randint(5, 50000, (4, 400)).cuda()
lab = torch.tensor([1., 0., 1., 0.]).cuda()
rep = check_gradient_flow(model, ids, lab,
                          loss_fn=lambda out, y: 0.7 * out[0] + 0.3 * contrastive(out[2], y))

assert rep['detached'] == [], f"вне графа автограда: {rep['detached']}"
print("\nобучаемых тензоров:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("из них LoRA:", sum(p.numel() for n, p in model.named_parameters()
                          if p.requires_grad and 'lora_' in n))
# lora_A получает нулевой градиент только пока lora_B нулевая, то есть на
# свежих адаптерах. У дообученного чекпоинта нулей быть не должно.
print("zero-grad на шаге 0:", rep['zero_grad'] or "нет — адаптеры уже обучены")
del model, enc; torch.cuda.empty_cache()

## 5. Плечи

`--pos_weight 4.0` соответствует отношению 1:4 в обучающей подвыборке. Без него
невзвешенный BCE уходит в «всё безопасно»: именно так получилась accuracy 0.928
при recall 0.231 в прежнем прогоне.

`--early_stopping_metric eval_pr_auc` — на 4.9 % позитивов останавливаться по
accuracy значит поощрять вырожденное решение.

Валидация и тест сохраняют естественную долю 4.9 %; перебалансирована только
обучающая часть. Поэтому вероятности откалиброваны под 1:4, и порог берётся
с валидации, а не 0.5.

Чекпоинты пишутся на Drive — сессия Colab обрывается, а `--init_from` при
наличии `checkpoint-last` уступает возобновлению, так что прерванный прогон
продолжится сам.

In [ ]:
%cd /content/GNN-ReGVD-Plus
SEED = 42

# Одной строкой намеренно: перенос строки внутри !-команды Colab не выполнит.
COMMON = ' '.join([
    '--do_train --do_eval --do_test --evaluate_during_training',
    f'--init_from {INIT}',
    f'--train_data_file {DATA}/megavul_train_sub.jsonl',
    f'--eval_data_file {DATA}/megavul_valid_15k.jsonl',
    f'--test_data_file {DATA}/megavul_test.jsonl',
    '--model_type roberta',
    '--model_name_or_path microsoft/graphcodebert-base',
    '--tokenizer_name microsoft/graphcodebert-base',
    '--block_size 400 --gnn ReGCN --num_classes 1',
    '--train_batch_size 16 --gradient_accumulation_steps 8 --eval_batch_size 64',
    '--learning_rate 1e-4 --epoch 4 --fp16 --num_workers 2',
    '--pos_weight 4.0 --freeze_encoder',
    '--early_stopping_patience 2 --early_stopping_metric eval_pr_auc',
    f'--checkpoint_format slim --seed {SEED}',
])
print(COMMON)

### Плечо B0 — контроль: адаптеры заморожены

In [ ]:
B0 = f'{DRIVE}/megavul_b0_seed{SEED}'
cmd = f'python code/run.py {COMMON} --freeze_lora --output_dir {B0}'
!{cmd} 2>&1 | tail -40

### Плечо B1 — адаптеры дообучаются

In [ ]:
B1 = f'{DRIVE}/megavul_b1_seed{SEED}'
cmd = f'python code/run.py {COMMON} --output_dir {B1}'
!{cmd} 2>&1 | tail -40

## 6. Оценка

Каждое число читается против своей точки отсчёта: PR-AUC против случайного
0.0492, accuracy против вырожденного 0.9508, и оба — против длины функции,
которая на zero-shot даёт PR-AUC 0.2087. Результат ниже этого уровня означает,
что модель не превзошла однострочную эвристику «чем длиннее, тем опаснее».

In [ ]:
import os
os.makedirs('/content/runs', exist_ok=True)

for name, out in (('B0', B0), ('B1', B1)):
    print('=' * 70); print(name); print('=' * 70)
    dump = f'/content/runs/megavul_{name}_seed{SEED}.jsonl'
    predict = (f'python code/predict_dump.py --model_path {out}/checkpoint-best-acc/model.bin '
               f'--input_file {DATA}/megavul_test.jsonl --output_file {dump} --batch_size 64')
    report = f'python code/report_metrics.py {dump} --paired'
    !{predict}
    !{report}

In [ ]:
import shutil, os
os.makedirs(f'{DRIVE}/runs', exist_ok=True)
for name in ('B0', 'B1'):
    src = f'/content/runs/megavul_{name}_seed{SEED}.jsonl'
    shutil.copy(src, f'{DRIVE}/runs/')
print("предсказания сохранены на Drive:", os.listdir(f'{DRIVE}/runs'))

## 7. Сиды

Один прогон ничего не доказывает — прежний вывод про LoRA держался на одном
сиде и потому не был защищён. Повторить разделы 5–6 с `SEED = 43` и `SEED = 44`,
затем сравнивать B1 − B0 как среднее ± σ по трём сидам.

Файлы предсказаний накапливаются в `MyDrive/gnn-regvd/runs/`; сведение по ним
считается локально.